# 07 — Does the assembled decoder learn?

A connected gradient is necessary but not sufficient evidence of learning. Here we test one bounded claim: can the tiny decoder memorize a consistent next-token batch?

**Pre-run contract:** seed 505; CPU float32; 2 sequences of length 7 shifted into [2,6]; vocabulary 16; width 16; two blocks; 4 heads of width 4; hidden width 32; 160 full-batch AdamW steps, learning rate .02, weight decay 0; no dropout, clipping, scheduler, or early stopping. Success: all losses/gradients finite, final CE < .05, training-token accuracy 100%. Do not change the seed or budget merely to hide a failure.

Specification: ../../experiments/specs/2026-09-06-decoder-notebooks.md.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


## 1. Audit the batch and training loop

Inspect the exact input/label pairs. What does a low loss on these pairs establish, and what remains untested? Read the small reusable training loop before running it.

**Your prediction:** _Write it here before running the reference._

In [ ]:
ids, labels = teaching_batch()
# Record your prediction here before training.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
sequence = torch.tensor([[1,2,3,4,5,6,7], [8,9,10,11,12,13,14]])
assert torch.equal(ids, sequence[:, :-1])
assert torch.equal(labels, sequence[:, 1:])
print("Input IDs:", ids)
print("Next-token labels:", labels)
print(inspect.getsource(fit_one_batch))

### Why this works

The labels are shifted exactly once and contain no conflicting next token for an identical prefix. Fitting them tests this implementation and optimization setup; it does not estimate language capability.

## 2. Run the declared learning experiment

Use the reference loop, or write your own matching the exact contract. Compare initial and final loss, inspect parameters, and do not stop at the first promising checkpoint.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Your implementation may go here; the adjacent reference runs the full contract.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
torch.manual_seed(505)
frozen = TinyDecoder().eval()
frozen_loss_before = float(next_token_loss(frozen(ids), labels).detach())
trained, result = fit_one_batch()
close(torch.tensor(result["initial_loss"]), torch.tensor(frozen_loss_before), atol=1e-6)
assert result["final_loss"] < .05 and result["accuracy"] == 1.
assert all(math.isfinite(value) for value in result["history"])
print({k: v for k, v in result.items() if k != "history"})
print("Loss samples:", [(i+1, result["history"][i]) for i in (0, 9, 39, 79, 159)])
print("Trained token predictions:", trained(ids).argmax(-1))
print("Embedding change norm:", float((trained.token.weight-frozen.token.weight).norm().detach()))

### Why this works

Each iteration computes logits, loss, gradients, and then an optimizer update. The final reported loss is evaluated after the last update; history entries are measured before each update. Success demonstrates one-batch learning, not generalization.

## 3. Freeze the control and test changed contexts

Keep the initial model unchanged. Change the order of the input tokens and inspect the trained model's predictions. Can the training accuracy tell us whether these new predictions are good?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict the evidence boundary before inspecting the changed-context outputs.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
frozen_loss_after = float(next_token_loss(frozen(ids), labels).detach())
assert frozen_loss_after == frozen_loss_before
changed = ids.flip(1)
with torch.no_grad():
    print("Changed input IDs:", changed)
    print("Changed-context predictions:", trained(changed).argmax(-1))
    print("Original-context predictions:", trained(ids).argmax(-1))
    prompt = ids[:1, :1].clone()
    for _ in range(6):
        next_id = trained(prompt)[:, -1].argmax(-1, keepdim=True)
        prompt = torch.cat((prompt, next_id), dim=1)
    print("Greedy rollout from the first training prefix:", prompt)
print("Frozen control loss stayed at:", frozen_loss_after)

### Why this works

The changed sequence has no declared gold continuation in this experiment, so its output is an observation, not an accuracy score. A familiar greedy rollout may succeed through memorization. To claim generalization, define an independent test task and evaluation contract first.

## Takeaway and evidence boundary

The baseline sequence is complete as material. Next, change one architectural mechanism at a time: RMSNorm/SwiGLU, RoPE, then GQA and cost accounting. Passing this notebook does not automatically establish learner mastery of each component.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.